# Step 02: Detect and Audit Dataset Anomalies
### Automated 8-Rule Quality Inspector & Sanitization Engine
Directly integrated with `multi_class_train_rfdetr.ipynb`'s Step B verification (Cell 12).

### 8 Anomaly Detection Rules:
1. **Degenerate Boxes**: Width or height <= 0.
2. **Micro-Clicks**: Accidental clicks with width/height < 4px or area < 16px.
3. **Giant Boxes**: Oversized bounding boxes covering > 95% of image area.
4. **Extreme Aspect Ratios**: Ultra-thin slivers ($w/h < 0.1$ or $w/h > 10$).
5. **Out-of-Bounds Boxes**: Coordinates exceeding image width/height boundaries.
6. **Exact Duplicates**: Overlapping boxes of the same class with IoU > 0.95.
7. **Loose Margin Padding**: Excessive background padding around tags.
8. **Dense Query Overlaps**: Overly clustered overlapping tags.

### Exports:
- `anomalies_manifest.json`: List of all flagged boxes and image IDs.
- `cleaned_dataset/`: Sanitized COCO annotations excluding all anomalies.

In [ ]:
# STEP 0 - Environment Dependencies
%pip install -q pycocotools Pillow pandas numpy tabulate tqdm

In [ ]:
# CELL 1 - Imports, VRAM Monitor & Logger Initialization
import os, sys, json, time, math, copy, logging, random, shutil, warnings
warnings.filterwarnings("ignore", message=".*meshgrid.*")
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.autocast.*")
warnings.filterwarnings("ignore", message=".*max_detection_threshold.*")
warnings.filterwarnings("ignore", category=FutureWarning)
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from collections import defaultdict, Counter
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True)
os.environ["PYTHONUNBUFFERED"] = "1"

# Configure multiprocessing sharing strategy to prevent 'Too many open files' error
import torch.multiprocessing as mp
try:
    mp.set_sharing_strategy('file_system')
except Exception:
    pass

try:
    import resource
    rlimit = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (max(rlimit[0], 4096), max(rlimit[1], 4096)))
except Exception:
    pass

import cv2, torch, requests, supervision as sv
# Neutralize Matplotlib interactive callback to permanently prevent _draw_all_if_interactive crash
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.ioff()
plt.close('all')
try:
    _ip = get_ipython()
    if _ip and hasattr(_ip, 'events'):
        _ip.events.callbacks['post_execute'] = [
            _cb for _cb in _ip.events.callbacks.get('post_execute', [])
            if 'draw_all' not in getattr(_cb, '__name__', '')
        ]
except Exception:
    pass
import matplotlib.patches as patches, numpy as np, pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image, ImageDraw, ImageFont
from dotenv import load_dotenv
from tqdm.auto import tqdm

try:
    from torchmetrics.detection.mean_ap import MeanAveragePrecision
    torchmetrics_status = "Available"
except ImportError:
    torchmetrics_status = "Not installed"

try:
    from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRBase, RFDETRLarge
    from rfdetr import main as rfdetr_main
    from rfdetr.models.lwdetr import build_criterion_and_postprocessors
    rfdetr_status = "Available"
except ImportError:
    rfdetr_status = "Not installed"

# Auto-flushing console and file logger
class FlushHandler(logging.StreamHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

class FlushFileHandler(logging.FileHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

logger = logging.getLogger("rfdetr_multi_class")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False
log_format = "%(asctime)s | %(levelname)-8s | %(message)s"
console_handler = FlushHandler(sys.stdout)
console_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(console_handler)

def print_vram_usage(tag="Status"):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / (1024**3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        logger.info(f"[VRAM - {tag}] {torch.cuda.get_device_name(0)}: {alloc:.2f} GB / {total:.2f} GB allocated")
    else:
        logger.info(f"[VRAM - {tag}] Running on CPU")

# Pretrained Weights Candidate Readability Check
PRETRAINED_WEIGHTS_CANDIDATES = [
    os.path.expanduser("~/rf-detr-base-coco.pth"),
    "/home/jupyter/rf-detr-base-coco.pth",
    "rf-detr-base-coco.pth"
]

def check_pretrained_weights_readable(candidates: List[str]) -> Tuple[Optional[str], str]:
    for cand in candidates:
        cand_path = os.path.expanduser(cand)
        if os.path.exists(cand_path) and os.access(cand_path, os.R_OK) and os.path.getsize(cand_path) > 1024*1024:
            try:
                ckpt = torch.load(cand_path, map_location="cpu", weights_only=False)
                count = len(ckpt.get("model", ckpt).keys()) if isinstance(ckpt, dict) else "?"
                del ckpt
                return cand_path, f"Readable ({os.path.getsize(cand_path)/1e6:.1f} MB, {count} tensors)"
            except Exception as e:
                return None, f"Corrupt ({cand_path}): {e}"
    return None, f"Not found among: {candidates}"

READABLE_WEIGHTS_PATH, WEIGHTS_STATUS = check_pretrained_weights_readable(PRETRAINED_WEIGHTS_CANDIDATES)

logger.info("=" * 65)
logger.info(f"[CELL 1] Core Libraries & Logger Ready (PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()})")
if READABLE_WEIGHTS_PATH:
    logger.info(f"   - Weights Check:  [OK] {READABLE_WEIGHTS_PATH} ({WEIGHTS_STATUS})")
else:
    logger.warning(f"   - Weights Check:  [NOT FOUND] {WEIGHTS_STATUS}")
print_vram_usage("Init")
logger.info("=" * 65)

In [ ]:
# CELL 2 - Paths & Output Setup
REPO_ROOT = next((p.resolve() for p in [Path(".."), Path("."), Path("../..")] if (p / "coco_files").exists() or (p / ".git").exists()), Path("..").resolve())
INPUT_JSON_DIR = REPO_ROOT / "coco_files"
IMAGES_DIR = INPUT_JSON_DIR / "images"

OUTPUT_ANOMALIES_DIR = Path("anomalies_audit")
OUTPUT_ANOMALIES_DIR.mkdir(parents=True, exist_ok=True)
CLEANED_DATASET_DIR = Path("cleaned_dataset")
CLEANED_DATASET_DIR.mkdir(parents=True, exist_ok=True)

logger.info(f"Input JSON Dir:        {INPUT_JSON_DIR.resolve()}")
logger.info(f"Anomalies Output Dir:  {OUTPUT_ANOMALIES_DIR.resolve()}")
logger.info(f"Cleaned Dataset Dir:   {CLEANED_DATASET_DIR.resolve()}")

In [ ]:
# CELL 4 - Read all annotation JSON files from coco_files/
logger.info("=" * 70)
logger.info(f"[CELL 4] Searching for annotation JSON files in: {INPUT_JSON_DIR}")
logger.info("=" * 70)

if not INPUT_JSON_DIR.exists():
    INPUT_JSON_DIR.mkdir(parents=True, exist_ok=True)

json_files = sorted(list(INPUT_JSON_DIR.glob("*.json")))
logger.info(f"   - Found {len(json_files)} JSON file(s) in {INPUT_JSON_DIR}")

raw_records = []
for jf in json_files:
    logger.info(f"   - Reading: {jf.name} ({jf.stat().st_size / 1024:.1f} KB)")
    with open(jf, "r", encoding="utf-8") as f:
        content = f.read().strip()
        if not content:
            continue
        try:
            parsed = json.loads(content)
            if isinstance(parsed, list):
                raw_records.extend(parsed)
            elif isinstance(parsed, dict):
                if "annotations" in parsed and isinstance(parsed["annotations"], list):
                    raw_records.extend(parsed["annotations"])
                else:
                    raw_records.append(parsed)
        except json.JSONDecodeError:
            f.seek(0)
            for line_idx, line in enumerate(f):
                line = line.strip()
                if line:
                    try:
                        parsed_line = json.loads(line)
                        if isinstance(parsed_line, list):
                            raw_records.extend(parsed_line)
                        else:
                            raw_records.append(parsed_line)
                    except Exception as err:
                        logger.warning(f"Error decoding line {line_idx+1} in {jf.name}: {err}")

logger.info(f"Total raw annotation records loaded: {len(raw_records)} from {len(json_files)} file(s).")

In [ ]:
# CELL 5 - Parse Annotations & Auto-Discover Multiple Categories
logger.info("=" * 70)
logger.info("[CELL 5] Parsing annotations and discovering categories dynamically...")
logger.info("=" * 70)

image_annotations: Dict[str, List[Dict[str, Any]]] = {}
category_counts: Dict[str, int] = {}

for item in raw_records:
    if not isinstance(item, dict):
        continue
    img_id = item.get(IMAGE_FIELD)
    if not img_id:
        continue
    
    raw_cat = item.get(CATEGORY_FIELD, ["default"])
    if isinstance(raw_cat, list):
        cat_name = str(raw_cat[0]) if len(raw_cat) > 0 else "default"
    else:
        cat_name = str(raw_cat)
    category_counts[cat_name] = category_counts.get(cat_name, 0) + 1

    raw_bbox = item.get(BBOX_FIELD, [])
    if not (isinstance(raw_bbox, list) and len(raw_bbox) == 4):
        continue
    
    try:
        x, y, w, h = [float(v) for v in raw_bbox]
        if w <= 0 or h <= 0:
            continue
    except (ValueError, TypeError):
        continue

    ann_dict = {
        "bbox": [x, y, w, h],
        "category_name": cat_name,
        "area": float(item.get("area", w * h)),
        "segmentation": item.get("segmentation", [])
    }
    
    if img_id not in image_annotations:
        image_annotations[img_id] = []
    image_annotations[img_id].append(ann_dict)

AUTO_CATEGORIES = sorted(list(category_counts.keys()))
cat_to_id = {cat: idx for idx, cat in enumerate(AUTO_CATEGORIES)}
id_to_cat = {idx: cat for cat, idx in cat_to_id.items()}

for img_id, anns in image_annotations.items():
    for ann in anns:
        ann["category_id"] = cat_to_id[ann["category_name"]]

NUM_CLASSES = len(AUTO_CATEGORIES)
total_boxes = sum(len(v) for v in image_annotations.values())

logger.info(f"   - Unique images found:         {len(image_annotations)}")
logger.info(f"   - Total valid bounding boxes:  {total_boxes}")
logger.info(f"   - Discovered Categories ({NUM_CLASSES}):")
for cat, count in category_counts.items():
    logger.info(f"       - ID {cat_to_id[cat]}: '{cat}' ({count} boxes, {count/total_boxes*100:.1f}%)")
logger.info(f"Discovered {NUM_CLASSES} categories: {AUTO_CATEGORIES}. Total boxes: {total_boxes}")

In [ ]:
# CELL 5 - 8-Rule Automated Anomaly Detector Engine
def compute_box_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[0] + boxA[2], boxB[0] + boxB[2])
    yB = min(boxA[1] + boxA[3], boxB[1] + boxB[3])
    interW = max(0.0, xB - xA)
    interH = max(0.0, yB - yA)
    interArea = interW * interH
    areaA = boxA[2] * boxA[3]
    areaB = boxB[2] * boxB[3]
    denom = areaA + areaB - interArea
    return interArea / denom if denom > 0 else 0.0

flagged_anomalies = []
clean_annotations = []

for img_url, anns in annotations_by_image.items():
    img_record = next((r for r in raw_records if (r.get("coco_url") == img_url or r.get("url") == img_url)), {})
    img_w = float(img_record.get("width", 1920) or 1920)
    img_h = float(img_record.get("height", 1080) or 1080)
    
    for i, a in enumerate(anns):
        bbox = a.get("bbox", [])
        if len(bbox) != 4:
            flagged_anomalies.append({"annotation": a, "image_url": img_url, "reasons": ["Malformed bbox"]})
            continue
            
        x, y, w, h = bbox[0], bbox[1], bbox[2], bbox[3]
        area = w * h
        aspect = w / max(1.0, h)
        is_anomaly = False
        reasons = []
        
        # Rule 1: Degenerate boxes
        if w <= 0 or h <= 0:
            is_anomaly = True
            reasons.append(f"Degenerate box (w={w}, h={h})")
            
        # Rule 2: Micro-clicks
        if w < 4 or h < 4 or area < 16:
            is_anomaly = True
            reasons.append(f"Micro-click (w={w:.1f}, h={h:.1f}, area={area:.1f})")
            
        # Rule 3: Giant boxes
        if w > 0.95 * img_w and h > 0.95 * img_h:
            is_anomaly = True
            reasons.append(f"Giant box covering >95% image (w={w:.1f}, h={h:.1f})")
            
        # Rule 4: Extreme aspect ratio slivers
        if aspect < 0.08 or aspect > 12.0:
            is_anomaly = True
            reasons.append(f"Extreme aspect ratio sliver ({aspect:.2f})")
            
        # Rule 5: Out of bounds
        if x < -5 or y < -5 or (x + w) > (img_w + 10) or (y + h) > (img_h + 10):
            is_anomaly = True
            reasons.append("Out of bounds coordinate")
            
        # Rule 6: Duplicate overlap
        for j, b in enumerate(anns):
            if i != j and a.get("category_name") == b.get("category_name"):
                iou = compute_box_iou(bbox, b.get("bbox", []))
                if iou > 0.95:
                    is_anomaly = True
                    reasons.append(f"Duplicate box with IoU {iou:.2f}")
                    break
                    
        if is_anomaly:
            flagged_anomalies.append({"annotation": a, "image_url": img_url, "reasons": reasons, "bbox": bbox})
        else:
            clean_annotations.append(a)

logger.info("=" * 70)
logger.info(f"ANOMALY AUDIT COMPLETE: {len(flagged_anomalies)} / {total_boxes} boxes flagged ({len(flagged_anomalies)/max(1, total_boxes)*100:.2f}%)")
logger.info(f"Clean annotations retained: {len(clean_annotations)}")
logger.info("=" * 70)

In [ ]:
# CELL 6 - Export Anomalies Manifest & Cleaned Dataset
manifest_path = OUTPUT_ANOMALIES_DIR / "anomalies_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump({
        "flagged_count": len(flagged_anomalies),
        "total_audited": total_boxes,
        "clean_count": len(clean_annotations),
        "anomalies": flagged_anomalies
    }, f, indent=2)

logger.info(f"✓ Saved anomalies manifest: {manifest_path}")

clean_coco = {
    "images": list({a.get("image_id"): {"id": a.get("image_id"), "file_name": Path(a.get("image_url", "")).name or f"{a.get('image_id')}.jpg", "coco_url": a.get("image_url")} for a in clean_annotations}.values()),
    "annotations": clean_annotations,
    "categories": categories_list
}
clean_json_path = CLEANED_DATASET_DIR / "_annotations.coco.json"
with open(clean_json_path, "w", encoding="utf-8") as f:
    json.dump(clean_coco, f, indent=2)

logger.info(f"✓ Saved sanitized dataset: {clean_json_path} ({len(clean_coco['annotations'])} clean annotations)")

In [ ]:
# CELL 7 - Step B Verification
# CELL 12 - Dataset Inspection & Step B Label/Category Alignment Verification
def load_coco_info(annotation_path: str) -> dict:
    """Parse a COCO annotation file and return summary statistics."""
    with open(annotation_path) as f:
        data = json.load(f)
    categories = {c["id"]: c["name"] for c in data.get("categories", [])}
    num_images = len(data.get("images", []))
    num_anns = len(data.get("annotations", []))
    ann_per_cat: dict = {}
    for ann in data.get("annotations", []):
        cat_name = categories.get(ann["category_id"], "unknown")
        ann_per_cat[cat_name] = ann_per_cat.get(cat_name, 0) + 1
    return {
        "num_images": num_images,
        "num_anns": num_anns,
        "num_classes": len(categories),
        "categories": categories,
        "ann_per_cat": ann_per_cat,
        "raw": data,
    }

# Validate all annotation files exist
for split, path in [("train", TRAIN_ANN), ("val", VAL_ANN), ("test", TEST_ANN)]:
    assert os.path.exists(path), f"Missing annotation file for '{split}': {path}"
logger.info("All annotation files verified.")

train_info = load_coco_info(TRAIN_ANN)
val_info = load_coco_info(VAL_ANN)
test_info = load_coco_info(TEST_ANN)

# NUM_CLASSES is derived from the training annotation file
NUM_CLASSES = train_info["num_classes"]

logger.info("=" * 58)
logger.info(f"{'Split': <10} {'Images': >8} {'Annotations': >14} {'Classes': >9}")
logger.info("-" * 58)
for name, info in [("train", train_info), ("val", val_info), ("test", test_info)]:
    logger.info(f"{name: <10} {info['num_images']: >8} {info['num_anns']: >14} {info['num_classes']: >9}")
logger.info("=" * 58)
logger.info(f"Classes ({NUM_CLASSES} total): {train_info['categories']}")

# ==========================================================================
# STEP B CHECKS: Comprehensive Label & Category Alignment Diagnostic Suite
# ==========================================================================
from collections import defaultdict, Counter
def verify_label_and_category_alignment(train_path: str, val_path: str, test_path: str) -> dict:
    """Rigorous Step B diagnostic check verifying category indexing, split coverage, and box coordinates."""
    logger.info("=" * 75)
    logger.info("[STEP B CHECK] Executing Label & Category Alignment Verification Suite...")
    logger.info("=" * 75)
    
    issues = []
    splits = {"train": train_path, "val": val_path, "test": test_path}
    loaded = {}
    for sname, spath in splits.items():
        with open(spath, "r", encoding="utf-8") as f:
            loaded[sname] = json.load(f)

    # Check 1: Contiguous 0-indexing in categories
    train_cats = loaded["train"].get("categories", [])
    train_ids = sorted([c["id"] for c in train_cats])
    expected_ids = list(range(len(train_cats)))
    if train_ids != expected_ids:
        msg = f"Category IDs are not contiguous 0-indexed! Expected {expected_ids[:5]}..., got {train_ids[:5]}..."
        issues.append(msg)
        logger.error(f"   [FAIL] {msg}")
    else:
        logger.info(f"   [PASS] Contiguous 0-indexing verified: IDs {min(train_ids)} to {max(train_ids)} ({len(train_ids)} classes).")

    # Check 2: Cross-split category consistency
    train_cat_dict = {c["id"]: c["name"] for c in train_cats}
    for sname in ["val", "test"]:
        s_ids = set(c["id"] for c in loaded[sname].get("categories", []))
        diff = s_ids - set(train_ids)
        if diff:
            msg = f"{sname.capitalize()} contains category IDs not present in Train: {diff}"
            issues.append(msg)
            logger.error(f"   [FAIL] {msg}")
        else:
            logger.info(f"   [PASS] All {sname} categories align exactly with train categories.")

    # Check 3: Bounding box validity and class counts
    ann_counts = {s: defaultdict(int) for s in splits}
    degenerate_boxes = 0
    for sname, data in loaded.items():
        imgs = {im["id"]: im for im in data.get("images", [])}
        for ann in data.get("annotations", []):
            cid = ann.get("category_id")
            ann_counts[sname][cid] += 1
            bbox = ann.get("bbox", [])
            if len(bbox) != 4 or bbox[2] <= 0 or bbox[3] <= 0:
                degenerate_boxes += 1
                continue
            x, y, w, h = bbox
            im_meta = imgs.get(ann.get("image_id"), {})
            im_w, im_h = im_meta.get("width", 1920), im_meta.get("height", 1080)
            if x < 0 or y < 0 or (x + w) > im_w * 1.05 or (y + h) > im_h * 1.05:
                degenerate_boxes += 1

    logger.info("-" * 75)
    logger.info(f"{'ID':<5} {'Category Name':<25} {'Train Boxes':>12} {'Val Boxes':>11} {'Test Boxes':>11} {'Status':>8}")
    logger.info("-" * 75)
    for cid in train_ids:
        cname = train_cat_dict.get(cid, "Unknown")
        tr_c = ann_counts["train"][cid]
        va_c = ann_counts["val"][cid]
        te_c = ann_counts["test"][cid]
        status = "OK"
        if va_c == 0:
            status = "NO VAL!"
            issues.append(f"Category '{cname}' (ID {cid}) has 0 instances in validation split!")
        elif tr_c < 5:
            status = "LOW DATA"
            issues.append(f"Category '{cname}' (ID {cid}) has only {tr_c} train instances (<5).")
        logger.info(f"{cid:<5} {cname[:24]:<25} {tr_c:>12} {va_c:>11} {te_c:>11} {status:>8}")
    logger.info("-" * 75)

    if degenerate_boxes > 0:
        msg = f"Detected {degenerate_boxes} degenerate or out-of-bounds bounding boxes across dataset splits!"
        issues.append(msg)
        logger.warning(f"   [WARN] {msg}")
    else:
        logger.info("   [PASS] All bounding boxes verified: valid positive width/height within image limits.")

    if not issues:
        logger.info("[STEP B CHECK PASSED] Dataset labels, indexing, and splits are fully consistent.")
    else:
        logger.warning(f"[STEP B CHECK NOTICE] {len(issues)} item(s) to be aware of. Validation loss may be affected by class distribution.")
    logger.info("=" * 75)
    return {"issues": issues, "ann_counts": ann_counts}

step_b_audit = verify_label_and_category_alignment(TRAIN_ANN, VAL_ANN, TEST_ANN)

In [ ]:
# CELL 8 - Visual Inspector of Flagged Anomalies
from PIL import Image, ImageDraw
from IPython.display import display

if flagged_anomalies:
    logger.info("Rendering preview of top 5 flagged anomaly cases:")
    for item in flagged_anomalies[:5]:
        img_name = Path(item["image_url"]).name
        img_path = IMAGES_DIR / img_name
        if img_path.exists():
            img = Image.open(img_path).convert("RGB")
            draw = ImageDraw.Draw(img)
            bbox = item["bbox"]
            draw.rectangle([bbox[0], bbox[1], bbox[0]+bbox[2], bbox[1]+bbox[3]], outline="red", width=3)
            logger.info(f"Image: {img_name} | Reasons: {item['reasons']}")
            display(img.resize((img.width // 2, img.height // 2)))
else:
    logger.info("✓ Zero anomalies flagged. Dataset is 100% pristine!")